# 🔄 Agent-to-Agent: Hierarchical Research with Bigdata.com

This notebook shows a **hierarchical agent** that checks internal data first, then escalates to the **Bigdata.com Research Agent** for deep, cited research when needed.

## What This Demonstrates

**Internal-first flow:**
- **Primary Agent** queries internal DB (portfolios, holdings, transactions) and internal research (FAISS vector store).
- **Escalation** to Bigdata.com Research Agent only when the question needs external, multi-source analysis (20–60s, full citations).

**Research Agent:**
- Multi-step reasoning with RAG; answers include inline citations as source-name hyperlinks (no citation numbers).
- Retry, stream timeout, and logging (production-ready; see `research_client.py`).

**Framework flexibility:**
> This demo uses **LangChain** and **LangSmith**. The pattern—internal tools first, then one “research” tool—works with **CrewAI**, **AutoGen**, or custom graphs. The key is tool ordering and system prompt that prefers internal sources.

## Architecture

![Agent to Bigdata Research Agent](./static/agent_to_research_small.jpg)

**Key benefits:** Cost and latency control (internal answers are fast); full citations when escalating; reusable `langgraph_core` setup and tools.

## Use Cases

| Role | Example Questions |
|------|-------------------|
| **Equity Research** | Thesis validation, competitive analysis |
| **Credit Research** | Covenant analysis, refinancing risks |
| **Credit Risk** | Counterparty exposure, default drivers |

---

## Langsmith Tracing

![LangSmith Tracing](./static/langsmith.png)

## 1️⃣ Install Dependencies

> **Note:** If you're using `uv` to manage dependencies (recommended), you can skip the pip install cell below. Dependencies are already installed via `uv pip install -r requirements.txt`.

In [1]:
# Dependencies are installed via uv: uv pip install -r requirements.txt
%pip install langchain langchain-openai langchain-community faiss-cpu requests python-dotenv -q

Note: you may need to restart the kernel to use updated packages.


## 2️⃣ Import Libraries

Import LangChain, display helpers, and reusable setup/tools from `langgraph_core`. Display helpers: `display_query`, `display_tools_used`, `display_response`, `display_citations`.

In [2]:
import os
from dotenv import load_dotenv
from IPython.display import display, Markdown, HTML

# LangChain
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent as langchain_create_agent

# Reusable core: environment, data sources, tools, display
import sys
sys.path.append(".")
from langgraph_core import (
    setup_environment,
    create_financial_database,
    create_vector_store,
    get_database_tools,
    get_vectorstore_tools,
    get_bigdata_tools,
    get_research_agent_tool,
    run_agent_query,
    display_query,
    display_response,
    display_tools_used,
    display_citations,
)
load_dotenv()
print("✅ Libraries imported successfully")

/Users/bakulkumarkakadiya/dev/github/bigdata-cookbook/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


✅ Libraries imported successfully


## 3️⃣ Setup Environment

In [3]:
# Setup with LangSmith tracing
config = setup_environment(
    langsmith_project='bigdata-agent-to-agent',
    enable_tracing=True
)
print("\n🔗 View agent traces: https://smith.langchain.com")

✅ LangSmith tracing enabled → Project: bigdata-agent-to-agent
✅ Bigdata API Key: bd_v2_ONWV...
✅ OpenAI API Key: sk-proj--w...

🔗 View agent traces: https://smith.langchain.com


## 4️⃣ Initialize Data Sources

Create the internal database and vector store with sample financial data.

In [4]:
# Create SQLite database with portfolios, holdings, transactions
create_financial_database()
# Create vector store with internal research documents
create_vector_store()
print("\n📊 Sample portfolios created:")
print("   • PF001: US Large Cap Growth (AAPL, META, AMZN, GOOGL, MSFT)")
print("   • PF002: AI & Semiconductor Focus (NVDA, AMD, AVGO, MRVL, TSM)")
print("   • PF003: Diversified Tech Leaders (NVDA, AAPL, MSFT, CRM, ORCL)")

✅ Created 3 accounts
✅ Created 3 portfolios
✅ Created 15 holdings
✅ Created 100 transactions
✅ Created vector store with 6 documents

📊 Sample portfolios created:
   • PF001: US Large Cap Growth (AAPL, META, AMZN, GOOGL, MSFT)
   • PF002: AI & Semiconductor Focus (NVDA, AMD, AVGO, MRVL, TSM)
   • PF003: Diversified Tech Leaders (NVDA, AAPL, MSFT, CRM, ORCL)


## 5️⃣ Create Hierarchical Agent

This agent is configured to:
1. Check **internal sources first** (faster, proprietary data)
2. Use **quick external lookups** for company info
3. **Escalate** to Research Agent only when needed

System prompt and agent are defined below so the flow is transparent.

In [5]:
# System prompt: internal-first, then escalate to Research Agent; inline citations as hyperlinks
HIERARCHICAL_SYSTEM_PROMPT = """You are a senior financial research analyst with access to both internal company systems and the Bigdata.com Research Agent for external research.

**IMPORTANT: Follow this research hierarchy:**

1. **ALWAYS check internal sources FIRST:**
   - `internal_query_database` - Check our portfolio positions, transactions, and account data
   - `internal_portfolio_summary` - Get quick overview of specific portfolios
   - `internal_search_research` - Search our internal research documents, investment theses, and analyst notes

2. **For entity resolution (optional):**
   - `bigdata_lookup_company` - Get Bigdata entity IDs for companies (useful for identification)

3. **ESCALATE to Research Agent when internal sources are insufficient:**
   - `bigdata_research_agent` - Use when you need deep analysis, market-wide trends, current events, or information internal sources cannot answer. Research Agent takes 20-60 seconds but provides comprehensive analysis with citations.

**Decision Framework:** Portfolio/holdings → Internal DB first. Company we hold → Internal research first, escalate if needed. Company we don't hold / market trends / news → Research Agent.

**Citation format:** When using Research Agent output, use inline citations with **only the source name as a hyperlink**—no citation numbers. Format as markdown: [Source Name](url). The reader should see only clickable source names (e.g. [Nasdaq](url), [Yahoo! Finance](url)); do not add numbers like [1], [2] in the text. Do not add a separate "Sources" or "References" block at the end—inline citations are sufficient. For internal data, briefly mention "From internal database" or "According to internal research."

**Available Portfolios:** PF001 (US Large Cap Growth), PF002 (AI & Semiconductor Focus), PF003 (Diversified Tech Leaders). Check internal sources first, then escalate to Research Agent for external data.
**Do not offer suggestions for follow up questions**
"""

# Build tools: internal first, then lookup, then research agent
tools = (
    get_database_tools() +
    get_vectorstore_tools() +
    [t for t in get_bigdata_tools() if t.name == "bigdata_lookup_company"] +
    get_research_agent_tool()
)
llm = ChatOpenAI(model="gpt-5", temperature=0, api_key=os.getenv("OPENAI_API_KEY"))
agent = langchain_create_agent(llm, tools, system_prompt=HIERARCHICAL_SYSTEM_PROMPT)
tool_names = [t.name for t in tools]
print(f"✅ Hierarchical agent created with {len(tools)} tools:")
print(f"   Internal: {[n for n in tool_names if n.startswith('internal_')]}")
print(f"   External: {[n for n in tool_names if n.startswith('bigdata_')]}")

✅ Hierarchical agent created with 5 tools:
   Internal: ['internal_query_database', 'internal_portfolio_summary', 'internal_search_research']
   External: ['bigdata_lookup_company', 'bigdata_research_agent']


---

# 📈 Equity Research Use Cases

Questions that equity analysts typically ask.

### Query 1: Internal Holdings Check (No Escalation Expected)

Simple portfolio question - should use only internal tools.

In [6]:
query = """
What is our total exposure to NVIDIA across all portfolios? 
Include position sizes, average cost basis, and unrealized P&L.
"""
display_query(query)
result = run_agent_query(agent, query, verbose=True, return_tools=True)
display_tools_used(result)
display_response(result)
# display_citations(result)

🔧 internal_query_database: {
  "sql_query": "SELECT h.portfolio_id, p.portfolio_name, h.ticker, h.company_name, h.shares, h.avg_cost, h.current_price, h.market_value, h.unrealiz...


From internal database.

By portfolio
- PF002 (AI & Semiconductor Focus): 12,000 shares of NVDA at $450.00 avg cost; market value $10,506,000 (at $875.50); unrealized P&L $5,106,000.
- PF003 (Diversified Tech Leaders): 8,000 shares of NVDA at $520.00 avg cost; market value $7,004,000 (at $875.50); unrealized P&L $2,844,000.
- PF001 (US Large Cap Growth): No position in NVIDIA.

Firmwide totals
- Total shares: 20,000
- Total cost basis: $9,560,000 (weighted avg cost $478.00/share)
- Total market value: $17,510,000 (at $875.50)
- Total unrealized P&L: $7,950,000

### Query 2: Investment Thesis Validation (Internal + External)

Combines internal research with market data - may escalate for current news.

In [7]:
query = """
Review our investment thesis for NVIDIA:
1. What does our internal research say about NVIDIA's competitive position?
2. What recent market developments might affect this thesis?
3. Should we adjust our position based on current information?
"""
display_query(query)
result = run_agent_query(agent, query, verbose=True, return_tools=True)
display_tools_used(result)
display_response(result)
# display_citations(result)

🔧 internal_portfolio_summary: {
  "portfolio_id": "PF001"
}...
🔧 internal_portfolio_summary: {
  "portfolio_id": "PF002"
}...
🔧 internal_portfolio_summary: {
  "portfolio_id": "PF003"
}...
🔧 internal_search_research: {
  "query": "NVIDIA NVDA competitive position moat CUDA software ecosystem data center AI accelerators competitors AMD MI300/MI325/MI350, Intel Gaudi...
🔧 internal_search_research: {
  "query": "NVIDIA networking Spectrum-X InfiniBand NVLink networking moat Mellanox acquisition, Ethernet vs InfiniBand roadmap, supply chain constr...
🔧 bigdata_research_agent: {
  "query": "Provide a concise, up-to-date brief (late-2025 to Jan-2026) on market developments relevant to NVIDIA\u2019s AI data center thesis. Focu...


Here’s a concise review of our NVIDIA thesis, combining our internal research with the latest externally sourced developments.

1) What our internal research says about NVIDIA’s competitive position
- Moat/software lock-in: Our thesis highlights CUDA’s entrenched ecosystem (4M+ developers) and NVIDIA’s full-stack software (CUDA, cuDNN, TensorRT, Triton, and emerging NIM microservices) as the core of switching costs and pricing power. According to internal research.
- Product leadership: Blackwell (B100/B200 and GB200 systems) expected to deliver ~2.5x performance improvement vs. Hopper and underpin the next leg of data center growth; inference TAM estimated at ~$150B by 2027. According to internal research.
- Networking integration: NVLink + InfiniBand/Spectrum-X integration seen as a strategic advantage after Mellanox, helping NVIDIA sell system-level solutions (DGX, NVL72). According to internal research.
- Risks flagged: China export controls (20–25% revenue at risk), supply constraints (TSMC advanced packaging, HBM), and rising competition (AMD, Intel Gaudi, hyperscaler custom silicon); valuation/AI “bubble” risk also noted. According to internal research.
- Portfolio stance: Strategy memo (Jan-2025) recommended increasing NVDA weight given demand outstripping supply; risk memo (Jan-2025) urged hedging given elevated valuations and China exposure. According to internal research.

2) Recent market developments that may affect the thesis
Product/roadmap and platform
- Blackwell ramping: GB300 systems are now shipping in large volumes, with reported software-driven training performance gains of up to 1.4x on the same hardware since initial launch (GB200 NVL72). See VentureBeat and Nasdaq: [VentureBeat](https://venturebeat.com/infrastructure/nvidias-vera-rubin-is-months-away-blackwell-is-getting-faster-right-now), [Nasdaq](https://www.nasdaq.com/articles/will-nvdas-blackwell-platform-support-its-data-center-revenue-growth).
- Demand signals: Commentary into Jan-2026 points to very strong Blackwell demand and resolution of some packaging bottlenecks; reports suggest NVIDIA booked a large share of CoWoS-L capacity for 2025–2026. [Miami Herald (MH)](https://www.miamiherald.com/news/business/article314365118.html#storylink=partnerdigest_the).
- Software and NIM: NVIDIA continues pushing AI Enterprise and NIM microservices to secure recurring software revenue alongside hardware. [PharmiWeb](https://www.pharmiweb.com/press-release/2025-12-15/computer-vision-in-healthcare-market-2030-size-share-growth-trends-and-forecast), [The Manila Times](https://www.manilatimes.net/2025/12/15/tmt-newswire/globenewswire/nvidia-debuts-nemotron-3-family-of-open-models/2243655).
- Networking strength: Networking revenue growth accelerated with Spectrum-X Ethernet, InfiniBand, and NVLink as hyperscalers build “AI factories.” [Futurum](https://futurumgroup.com/insights/nvidia-q3-fy-2026-record-data-center-revenue-higher-q4-guide/). Co-packaged optics adoption in Spectrum-X/Quantum-X expected to expand into 2026. [Intelligent Living](https://www.intelligentliving.co/photonic-chips-data-center-networking/).

Competition and hyperscaler silicon
- AMD: Street checks indicate strong hyperscaler demand into 2026; AMD’s server CPU business is projected to grow at least 50% in 2026 with continued MI4xx roadmap cadence. [Business Insider](https://markets.businessinsider.com/news/stocks/it-s-time-to-pounce-says-john-vinh-on-amd-stock-1035710206), [HotHardware](https://hothardware.com/news/instinct-mi400-challenge-vera-rubin).
- Google TPU: TPU v5e/v5p/Trillium (and Ironwood) expanding; Anthropic plans to access up to one million TPUs in 2026; discussions reported about a TPU deal with Meta. [Hindustan Times](https://www.hindustantimes.com/technology/neural-dispatch-stressful-times-for-nvidia-s-world-view-and-ai-refuses-to-take-responsibility-101764707676594.html), [Forbes.com](https://www.forbes.com/sites/johnwerner/2025/12/10/the-rise-of-the-tpu-innovating-for-specialization/), [Yahoo! Finance](https://finance.yahoo.com/news/nvidia-blackwell-vs-google-tpu-154032183.html), [EconoTimes.com](https://www.econotimes.com/Google-Accelerates-AI-Infrastructure-With-Ironwood-TPU-Expansion-in-2026-1729916).
- Microsoft Maia 200: Launched Jan-2026 for inference (TSMC 3nm), with ramp expected in 2H26; Microsoft still intends to buy NVIDIA/AMD alongside its own chips. [TechCrunch](https://techcrunch.com/2026/01/29/microsoft-wont-stop-buying-ai-chips-from-nvidia-amd-even-after-launching-its-own-nadella-says/), [TechCrunch](https://techcrunch.com/2026/01/26/microsoft-announces-powerful-new-chip-for-ai-inference/), [Financial Express](https://www.financialexpress.com/life/technology-microsoft-rolls-out-maia-200-to-take-on-nvidia-ai-chips-software-ai-here-is-all-you-need-to-know-4120820/), [Yahoo! Finance](https://uk.finance.yahoo.com/news/marvell-stock-rises-broadcom-hopes-174950288.html).

Supply chain and packaging
- Advanced packaging: CoWoS-L availability was a key gating factor in 2025; reports in Jan-2026 point to improved availability and NVIDIA pre-bookings, supporting the Blackwell ramp. [Miami Herald (MH)](https://www.miamiherald.com/news/business/article314365118.html#storylink=partnerdigest_the).

China export controls
- H200 to China: Preliminary U.S. approval was followed by Chinese customs blocking shipments in Jan-2026, creating uncertainty; any resolution would be incremental to prior outlook. [Yahoo! Finance](https://uk.finance.yahoo.com/news/china-blocks-nvidias-h200-shipments-113514733.html), [Yahoo! Finance](https://finance.yahoo.com/news/nvidia-nvda-ceo-jensen-huang-142926253.html), [MSN](https://www.msn.com/en-us/money/markets/nvidia-nvda-ceo-jensen-huang-frames-ai-as-the-largest-infrastructure-buildout-in-human-history/ar-AA1UT0uo?ocid=finance-verthp-feeds), [Nasdaq](https://www.nasdaq.com/articles/nvidia-investors-just-got-incredible-news-2026).

Financial and demand signals
- Ongoing strength in data center and networking with a higher guide cited in Q3 FY26 commentary; Loop Capital also projected NVIDIA could double GPU unit shipments into early 2026. [Futurum](https://futurumgroup.com/insights/nvidia-q3-fy-2026-record-data-center-revenue-higher-q4-guide/), [Yahoo! Finance](https://finance.yahoo.com/news/wall-street-loves-qualcomm-nvidia-153132055.html).
- Customer deployments: Examples of large B200/Blackwell deployments continue to be announced. [Associated Press](https://apnews.com/press-release/business-wire/sharon-ai-to-deploy-1k-b200-cluster-at-nextdc-m3-data-center-using-lenovo-infrastructure-and-vast-data-de83bd8835ef482784f319dfd7b7b543).

3) Should we adjust our position now?
Context from internal database:
- PF002 (AI & Semis): NVDA is ~$10.5M of ~$16.0M total MV (~66% single-name concentration). From internal database.
- PF003 (Diversified Tech): NVDA is ~$7.0M of ~$23.2M (~30%). From internal database.
- PF001 has smaller or tactical NVDA exposure. From internal database.

Recommendation
- Maintain an Overweight in NVDA based on sustained product leadership (Blackwell ramp, platform-level networking/software leverage) and continued hyperscaler demand, while acknowledging rising competition and China risk.
- Risk-manage the concentration:
  - PF002 (Aggressive): Trim 10–15% of the NVDA position to bring single-name exposure closer to 50% max without changing the core Overweight. Use staged trims into strength or ahead of event risk (earnings/China license headlines). From internal risk guidance.
  - PF003 (Moderate): Keep NVDA around a 25–30% cap to balance upside with diversification; trim modestly if it drifts above 30%.
  - PF001: Maintain current smaller weight; add only on dislocations tied to supply/China headlines if risk budget allows.
- Hedging: Consistent with prior internal risk guidance, maintain portfolio hedges during event windows (e.g., index put spreads) and consider collars on a slice of the NVDA position if implied volatility is favorable. According to internal research.

Key near-term catalysts and watch items
- Positive: Blackwell/GB300 volume milestones and software performance gains; continued networking outperformance; confirmation that packaging capacity remains ample; visible hyperscaler deployments. [VentureBeat](https://venturebeat.com/infrastructure/nvidias-vera-rubin-is-months-away-blackwell-is-getting-faster-right-now), [Nasdaq](https://www.nasdaq.com/articles/will-nvdas-blackwell-platform-support-its-data-center-revenue-growth), [Futurum](https://futurumgroup.com/insights/nvidia-q3-fy-2026-record-data-center-revenue-higher-q4-guide/).
- Risks: China import clearance for H200 remains uncertain; increasing deployments of hyperscaler custom silicon (TPUs, Maia) and AMD MI4xx ramps; valuation sensitivity into earnings. [Yahoo! Finance](https://uk.finance.yahoo.com/news/china-blocks-nvidias-h200-shipments-113514733.html), [TechCrunch](https://techcrunch.com/2026/01/29/microsoft-wont-stop-buying-ai-chips-from-nvidia-amd-even-after-launching-its-own-nadella-says/), [Hindustan Times](https://www.hindustantimes.com/technology/neural-dispatch-stressful-times-for-nvidia-s-world-view-and-ai-refuses-to-take-responsibility-101764707676594.html), [Business Insider](https://markets.businessinsider.com/news/stocks/it-s-time-to-pounce-says-john-vinh-on-amd-stock-1035710206), [HotHardware](https://hothardware.com/news/instinct-mi400-challenge-vera-rubin).

Bottom line: Our core thesis—NVIDIA’s end-to-end platform advantage across silicon, networking, and software—remains intact and supported by recent developments. We recommend maintaining the Overweight while trimming PF002’s outsized concentration and keeping PF003 within a 25–30% band, with hedges around key events.

### Query 3: Competitive Analysis (Escalation Expected)

Deep competitive analysis requires external research agent.

In [ ]:
query = """
Perform a competitive analysis of AMD vs NVIDIA in the AI accelerator market:

1. First check: Do we have any internal research comparing these companies?
2. Then analyze: Current Market share trends, product roadmaps, and customer wins
3. Conclude: Which company is better positioned for 2025-2026?

Cite specific sources for your analysis.
"""
display_query(query)
result = run_agent_query(agent, query, verbose=True, return_tools=True)
display_tools_used(result)
display_response(result)
# # display_citations(result)

🔧 internal_search_research: {
  "query": "AMD vs NVIDIA competitive analysis in AI accelerators (H100/H200/Blackwell vs MI300/MI325/MI350), CUDA vs ROCm, market share, customer w...
🔧 bigdata_research_agent: {
  "query": "Provide a concise, source-backed competitive analysis of AMD vs NVIDIA in the AI accelerator (data center GPU) market, focusing on 2024-...
🔧 bigdata_research_agent: {
  "query": "Collect authoritative citations for AMD vs NVIDIA AI accelerator market analysis (2024-2026): 1) Market share of AI/data center GPUs (Tr...
🔧 bigdata_research_agent: {
  "query": "Find authoritative 2024-2026 sources on AI accelerator (data center GPU) market for AMD vs NVIDIA: 1) Market share/trends: TrendForce AI...


---

# 💳 Credit Research Use Cases

Questions relevant to fixed income and credit analysts.

### Query 4: Debt and Leverage Analysis (Escalation Expected)

Credit-focused questions typically need external data.

In [ ]:
query = """
Analyze the credit profile of Intel Corporation:

1. Current debt levels and maturity schedule
2. Cash flow coverage ratios and liquidity position
3. Recent credit rating actions or outlook changes
4. Key risks that could affect their investment-grade status

Note: We don't hold Intel, so you may need external sources.
"""
display_query(query)
result = run_agent_query(agent, query, verbose=True, return_tools=True)
display_tools_used(result)
display_response(result)
# display_citations(result)

### Query 5: Sector-Wide Credit Trends (Escalation Expected)

Macro credit analysis requires deep research.

In [ ]:
query = """
What are the key credit risks facing the semiconductor sector in the current environment?

Consider:
- Capital expenditure requirements and debt financing
- Cyclical demand patterns and inventory corrections
- Geopolitical risks (US-China tensions, export controls)
- Recent bond issuances or refinancing activity

Which semiconductor companies are most vulnerable from a credit perspective?
"""
display_query(query)
result = run_agent_query(agent, query, verbose=True, return_tools=True)
display_tools_used(result)
display_response(result)
# display_citations(result)

### Query 6: Refinancing Risk Assessment

In [ ]:
query = """
Which companies in our portfolios might face refinancing challenges in 2025-2026?

1. First, check what companies we hold across all portfolios
2. For each major holding, assess:
   - Debt maturity walls
   - Current interest coverage
   - Access to capital markets
3. Flag any companies with elevated refinancing risk
"""
display_query(query)
result = run_agent_query(agent, query, verbose=True, return_tools=True)
display_tools_used(result)
display_response(result)
# display_citations(result)

---

# ⚠️ Credit Risk Analyst Use Cases

Questions focused on counterparty risk and default analysis.

### Query 7: Counterparty Exposure Analysis (Internal First)

In [ ]:
query = """
Calculate our total counterparty exposure by company:

1. Sum up all positions across portfolios for each ticker
2. Calculate as percentage of total AUM
3. Identify our top 5 concentration risks
4. Flag any single-name exposures exceeding 15% of total

Present in a risk report format.
"""
display_query(query)
result = run_agent_query(agent, query, verbose=True, return_tools=True)
display_tools_used(result)
display_response(result)
# display_citations(result)

### Query 8: ESG/Regulatory Risk (Escalation Expected)

In [ ]:
query = """
Assess regulatory and ESG risks for our technology holdings:

1. Which of our holdings face significant regulatory scrutiny?
2. What recent regulatory developments could impact valuations?
3. Are there any ESG controversies affecting our portfolio companies?
4. Which positions should we consider reducing due to regulatory risk?

Focus on antitrust, data privacy, and AI governance regulations.
"""
display_query(query)
result = run_agent_query(agent, query, verbose=True, return_tools=True)
display_tools_used(result)
display_response(result)
# display_citations(result)

---

# 🔬 Custom Query

Try your own hierarchical research query:

In [ ]:
# Enter your own query
custom_query = """
Compare our internal research on NVIDIA with the latest market sentiment.
Is our thesis still valid?
"""
display_query(custom_query)
result = run_agent_query(agent, custom_query, verbose=True, return_tools=True)
display_tools_used(result)
display_response(result)
# display_citations(result)

---

## 📊 Observability

View detailed traces in LangSmith:
- Each query shows the **tool call sequence**
- See which tools were checked first vs escalated
- Monitor **latency** differences between internal vs external calls
- Track **token usage** for cost optimization

**Dashboard:** https://smith.langchain.com


## 🔟 Key Benefits of This Architecture

1. **Internal-first** — Fast answers from DB and vector store; escalate only when external research is needed.
2. **Full citations** — Research Agent returns inline [1], [2] and numbered sources; display via `display_citations()`.
3. **Production-ready** — Research client has retry, stream timeout, and full chat_id logging.
4. **Reusability** — `create_financial_database()`, `create_vector_store()`, and tools come from `langgraph_core.py`.
5. **Observability** — LangSmith traces show when the agent uses internal vs Research Agent tools.

## 🎯 Next Steps

- **Add Search API** — Combine with `get_bigdata_tools()` (Search + KG) for news/filings before escalating to Research Agent (see `agent_to_search.ipynb`).
- **Tune escalation** — Adjust system prompt so the agent escalates only for “deep research” or “recent market” questions.
- **Research effort** — Use `research_effort="lite"` for faster, lighter answers or `"standard"` for full depth.
- **Follow-up** — Use `result.chat_id` and `client.follow_up()` for multi-turn Research Agent conversations.
- **LangSmith** — Use traces to see tool order and Research Agent usage.

---

## 📚 Additional Resources

- **Bigdata.com API docs**: https://docs.bigdata.com
- **Research Agent client**: `Research_Agent_Sync_Response/README.md` (retry, logging, chat_id)
- **LangGraph / LangChain**: https://langchain-ai.github.io/langgraph/
- **LangSmith**: https://smith.langchain.com
- **Agent_To_BigData README**: `README.md` in this folder

**Questions?** support@bigdata.com